# Неделя 2 — Сборка обучающей выборки

Задание: `docs/week2_dataset.md`

## 1. Определение таргета

TODO: зафиксируйте здесь текстом определение оттока, окна наблюдения и прогноза (и продублируйте в журнале).

- Отток - это когда клиент уходит или просто ПЕРЕСТАËТ пользоваться услугами.
- Окно наблюдение - это промежуток времени до точки отсчета, который мы берем для определения признаков.
- Окно прогноза - это промежуток времени после точки отсчета, по которому мы определяем target.

- Объект наблюдения: B2B-клиент (client_id) в точке отсчёта (snapshot — месяц в формате YYYY-MM).
- Окно наблюдения: 6 месяцев, включая месяц снапшота. Только из него считаем признаки.
- Окно прогноза: 3 месяца после месяца снапшота. Только из него считаем таргет.
- Событие оттока (target = 1): клиент ушёл, если в окне прогноза выполнилось хотя бы одно условие:
  1) явный отток: termination_date попадает в окно прогноза;
  2) тихий отток: суммарный revenue за 3 месяца окна равен 0 (перестал пользоваться услугами, не расторгая договор).
- Иначе target = 0.
- В выборку на каждый снапшот берём только "живых": подключился до снапшота, не расторгнут до снапшота, revenue > 0 в месяц снапшота (фильтр "мёртвых душ").

In [28]:
import pandas as pd
clients = pd.read_parquet('../data/processed/clients.parquet')
usage = pd.read_parquet('../data/processed/usage.parquet')

## 2. Сборка по нескольким snapshot'ам

In [29]:
snapshots = ['2024-07', '2024-10', '2025-01', '2025-04', '2025-07', '2025-10']
dataset = pd.concat([build_snapshot(usage, clients, s) for s in snapshots], ignore_index=True)
dataset.to_parquet('../data/processed/dataset.parquet')


## 3. Обязательные проверки

In [30]:
print(dataset.groupby('snapshot_date')['target'].agg(['mean', 'count']))  # стабильность доли оттока


                   mean  count
snapshot_date                 
2024-07        0.029216  31900
2024-10        0.031969  33783
2025-01        0.037888  35394
2025-04        0.044922  36819
2025-07        0.049069  37987
2025-10        0.052865  38797


In [33]:
print(dataset['segment'].isna().mean())  # нет пропущенного сегмента


0.0


In [35]:
assert not dataset.duplicated(subset=['client_id', 'snapshot_date']).any()


## 4. Самопроверка на утечку

TODO: по каждому признаку письменно ответьте — мог ли я знать это значение в 00:00 дня snapshot?

Мог знать в момент снапшота:
- revenue_mean_6m, revenue_last — выручка за месяцы окна наблюдения
- traffic_mean_6m — трафик за те же месяцы
- tickets_sum_6m — обращения в поддержку до снапшота
- debt_max_6m — долг на концы месяцев окна наблюдения
- n_sim_last — число SIM в месяц снапшота
- segment, product, region — статичные атрибуты, не меняются во времени
- client_id, snapshot_date — идентификатор и точка отсчёта

Не мог знать
- current_status — срез на июнь 2026, информация из будущего;
- termination_date — используем только для таргета из окна прогноза (это законно: таргет и обязан быть из будущего), но никогда как признак.